## Example 1: starlette functionality test 1

In [ ]:
# Example 1: Basic Application
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient

print("=== Basic Application ===")

async def homepage(request):
    return JSONResponse({"message": "Hello Starlette"})

app = Starlette(routes=[Route('/', homepage)])
client = TestClient(app)

response = client.get("/")
print(f"Response: {response.json()}")
print(f"Status: {response.status_code}")
assert response.status_code == 200, response.status_code
assert response.json() == {"message": "Hello Starlette"}, response.json()
missing = client.get("/no-such-page")
assert missing.status_code == 404, f"an unrouted path returned {missing.status_code}, not 404"

print(f"\nUse case: ASGI web framework")

print("\nExample 1 completed!")

## Example 2: starlette functionality test 2

In [ ]:
# Example 2: Middleware
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.cors import CORSMiddleware
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient

print("=== Middleware ===")

async def endpoint(request):
    return JSONResponse({"data": "test"})

middleware = [Middleware(CORSMiddleware, allow_origins=['*'])]
app = Starlette(routes=[Route('/', endpoint)], middleware=middleware)
client = TestClient(app)

# CORSMiddleware only answers a cross-origin request, so the request must carry an Origin header
response = client.get("/", headers={"Origin": "https://a.example"})
print(f"CORS headers: {response.headers.get('access-control-allow-origin')}")
assert response.status_code == 200 and response.json() == {"data": "test"}, (response.status_code, response.text)
assert response.headers.get('access-control-allow-origin') == '*', dict(response.headers)
print(f"\nUse case: Add CORS middleware")

print("\nExample 2 completed!")

## Example 3: starlette functionality test 3

In [ ]:
# Example 3: Path Parameters
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient

print("=== Path Parameters ===")

async def user_handler(request):
    user_id = request.path_params['user_id']
    return JSONResponse({"user_id": user_id})

app = Starlette(routes=[Route('/users/{user_id}', user_handler)])
client = TestClient(app)

response = client.get("/users/123")
print(f"Response: {response.json()}")
print(f"Captured user_id: {response.json()['user_id']}")
assert response.status_code == 200 and response.json() == {"user_id": "123"}, (response.status_code, response.text)

print(f"\nUse case: Dynamic URL routing")

print("\nExample 3 completed!")

## Example 4: starlette functionality test 4

In [ ]:
# Example 4: Request Methods
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient

print("=== Request Methods ===")

async def api_handler(request):
    return JSONResponse({"method": request.method})

app = Starlette(routes=[Route('/api', api_handler, methods=['GET', 'POST'])])
client = TestClient(app)

get_response = client.get("/api")
print(f"GET: {get_response.json()}")

post_response = client.post("/api")
print(f"POST: {post_response.json()}")
assert get_response.status_code == 200 and get_response.json() == {"method": "GET"}, get_response.text
assert post_response.status_code == 200 and post_response.json() == {"method": "POST"}, post_response.text
# A method the route does not declare is refused
put_response = client.put("/api")
assert put_response.status_code == 405, f"PUT on a GET/POST route returned {put_response.status_code}, not 405"

print(f"\nUse case: Handle multiple HTTP methods")

print("\nExample 4 completed!")

## Example 5: starlette functionality test 5

In [ ]:
# Example 5: Background Tasks
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route
from starlette.testclient import TestClient
from starlette.background import BackgroundTask

print("=== Background Tasks ===")

executed = []

def log_task(message):
    executed.append(message)
    print(f"Background: {message}")

async def task_handler(request):
    task = BackgroundTask(log_task, message="Task executed")
    return JSONResponse({"status": "queued"}, background=task)

app = Starlette(routes=[Route('/task', task_handler)])
client = TestClient(app)

response = client.get("/task")
print(f"Response: {response.json()}")
assert response.status_code == 200 and response.json() == {"status": "queued"}, response.text
# TestClient runs the response's background task before it returns
assert executed == ["Task executed"], executed

print(f"\nUse case: Execute background tasks")

print("\nExample 5 completed!")